# Areal (km² og %) av grunn sjø (≤ 25 m dyp) per bunntype for hver økoregion

For hver polygon i `Okoregioner_4stk.geojson`:

- Kutt DEM til regionen (`gdalwarp -cutline`).
- Rasteriser substratpolygonene (fastbunn / løsbunn) på nøyaktig samme grid (`gdal_rasterize`).
- Tell piksler der `-25 ≤ z ≤ 0` og substrat er hard- eller løsbunn.
- Pikselareal = 50 × 50 m = 2500 m².

In [12]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal, ogr, osr

gdal.UseExceptions()
ogr.UseExceptions()
gdal.SetConfigOption('OGR_GEOJSON_MAX_OBJ_SIZE', '0')

from pyogrio import write_dataframe

REPO = Path('/home/kim/work/marint-naturkart-nivaR')
DEM_PATH = REPO / 'features/feature_norge_dem50_depth_filled.tif'
REGIONS_PATH = REPO / 'notebooks/scratch/Okoregioner_4stk.geojson'
SUBSTRAT_PATH = REPO / 'notebooks/01_substrat/nisjedata-substrat_norge_2026_25833.geo.parquet'

DEPTH_THRESHOLDS = [25.0, 30.0]     # produserer én CSV per terskel
PIXEL_AREA_M2 = 50 * 50
TARGET_CRS = 'EPSG:25833'           # DEM & substrat CRS

BT_CODE = {'løsbunn': 1, 'fastbunn': 2}
BT_LABEL = {1: 'losbunn', 2: 'hardbunn'}

LM_CODE = {'DK_AB, DK_C, DK_D': 1, 'DK_EFGY': 2, 'DK_0': 3}
LM_LABEL = {1: 'dk-abcd', 2: 'dk-efgy', 3: 'dk-0'}

In [13]:
regions = gpd.read_file(REGIONS_PATH).to_crs(TARGET_CRS)
print(regions[['ID', 'Navn', 'Region']])

substrat = gpd.read_parquet(SUBSTRAT_PATH, columns=['BunnType', 'LM_DK', 'geometry'])
substrat['code_bt'] = substrat['BunnType'].map(BT_CODE).fillna(0).astype('int16')
substrat['code_lm'] = substrat['LM_DK'].map(LM_CODE).fillna(0).astype('int16')
substrat = substrat[(substrat['code_bt'] > 0) | (substrat['code_lm'] > 0)].copy()
if substrat.crs is None or substrat.crs.to_string() != TARGET_CRS:
    substrat = substrat.to_crs(TARGET_CRS)
substrat.sindex
print('substrat features:', len(substrat))

   ID          Navn          Region
0   0   Norskehavet   3_Norskehavet
1   0     Skagerrak     1_Skagerrak
2   0     Nordsjøen     2_Nordsjøen
3   0  Barentshavet  4_Barentshavet
substrat features: 440276


In [14]:
dem_ds = gdal.Open(str(DEM_PATH))
gt = dem_ds.GetGeoTransform()
px, py = gt[1], -gt[5]
x0, y0 = gt[0], gt[3]  # upper-left
dem_nodata = dem_ds.GetRasterBand(1).GetNoDataValue()
print('pixel size:', px, py, 'nodata:', dem_nodata)

def snap_bbox(minx, miny, maxx, maxy):
    """Snap a bbox outward to the DEM 50 m grid."""
    sx0 = x0 + np.floor((minx - x0) / px) * px
    sx1 = x0 + np.ceil((maxx - x0) / px) * px
    sy1 = y0 - np.floor((y0 - maxy) / py) * py
    sy0 = y0 - np.ceil((y0 - miny) / py) * py
    return sx0, sy0, sx1, sy1

pixel size: 50.0 50.0 nodata: 99999.0


In [15]:
def clip_dem(region_gdf, te, out_vsimem):
    """gdalwarp DEM to region bbox, cut to polygon."""
    mem_cutline = f'/vsimem/{Path(out_vsimem).stem}_cut.fgb'
    write_dataframe(region_gdf[['geometry']], mem_cutline, driver='FlatGeobuf')
    ds = gdal.Warp(
        out_vsimem, str(DEM_PATH),
        format='GTiff',
        outputBounds=te,
        xRes=px, yRes=py,
        targetAlignedPixels=False,
        cutlineDSName=mem_cutline,
        cropToCutline=False,
        dstNodata=dem_nodata,
        multithread=True,
        warpOptions=['NUM_THREADS=ALL_CPUS'],
        creationOptions=['TILED=YES', 'COMPRESS=LZW', 'BIGTIFF=IF_SAFER'],
    )
    gdal.Unlink(mem_cutline)
    return ds


def rasterize_substrat(sub_gdf, te, out_vsimem, attribute):
    """Rasterize substrat polygons (given attribute) onto the DEM grid."""
    mem_vec = f'/vsimem/{Path(out_vsimem).stem}_{attribute}.fgb'
    sel = sub_gdf[sub_gdf[attribute] > 0][[attribute, 'geometry']]
    write_dataframe(sel, mem_vec, driver='FlatGeobuf')
    minx, miny, maxx, maxy = te
    width = int(round((maxx - minx) / px))
    height = int(round((maxy - miny) / py))
    drv = gdal.GetDriverByName('GTiff')
    ds = drv.Create(out_vsimem, width, height, 1, gdal.GDT_Byte,
                    options=['TILED=YES', 'COMPRESS=LZW'])
    ds.SetGeoTransform((minx, px, 0, maxy, 0, -py))
    srs = osr.SpatialReference(); srs.ImportFromEPSG(25833)
    ds.SetProjection(srs.ExportToWkt())
    ds.GetRasterBand(1).SetNoDataValue(0)
    ds.GetRasterBand(1).Fill(0)
    gdal.Rasterize(ds, mem_vec, attribute=attribute, allTouched=False)
    gdal.Unlink(mem_vec)
    return ds

In [16]:
to_km2 = PIXEL_AREA_M2 / 1e6
results_by_thr = {t: [] for t in DEPTH_THRESHOLDS}

for _, row in regions.iterrows():
    name = row['Navn']
    print(f'--- {name} ---')
    region_gdf = gpd.GeoDataFrame([row], crs=regions.crs)
    minx, miny, maxx, maxy = region_gdf.total_bounds
    te = snap_bbox(minx, miny, maxx, maxy)

    tag = f'reg_{int(row["ID"])}_{name}'
    dem_out = f'/vsimem/{tag}_dem.tif'
    bt_out = f'/vsimem/{tag}_bt.tif'
    lm_out = f'/vsimem/{tag}_lm.tif'

    dem_ds_r = clip_dem(region_gdf, te, dem_out)
    dem_arr = dem_ds_r.GetRasterBand(1).ReadAsArray()
    dem_nd = dem_ds_r.GetRasterBand(1).GetNoDataValue()
    dem_ds_r = None

    idx = list(substrat.sindex.intersection((minx, miny, maxx, maxy)))
    sub_sel = substrat.iloc[idx]
    print(f'  substrat features in bbox: {len(sub_sel)}')

    bt_ds = rasterize_substrat(sub_sel, te, bt_out, attribute='code_bt')
    bt_arr = bt_ds.GetRasterBand(1).ReadAsArray(); bt_ds = None
    lm_ds = rasterize_substrat(sub_sel, te, lm_out, attribute='code_lm')
    lm_arr = lm_ds.GetRasterBand(1).ReadAsArray(); lm_ds = None

    valid = dem_arr != dem_nd

    for thr in DEPTH_THRESHOLDS:
        shallow = valid & (dem_arr >= -thr) & (dem_arr <= 0.0)
        n_shallow = int(shallow.sum())
        rec = {'ID': int(row['ID']), 'Navn': name,
               f'above_{int(thr)}m_total_km2': n_shallow * to_km2}
        pcts = {}
        for code, lab in BT_LABEL.items():
            n = int(np.count_nonzero(shallow & (bt_arr == code)))
            rec[f'{lab}_km2'] = n * to_km2
            pcts[f'{lab}_pct_of_above_{int(thr)}m'] = 100 * n / n_shallow if n_shallow else np.nan
        for code, lab in LM_LABEL.items():
            n = int(np.count_nonzero(shallow & (lm_arr == code)))
            rec[f'{lab}_km2'] = n * to_km2
            pcts[f'{lab}_pct_of_above_{int(thr)}m'] = 100 * n / n_shallow if n_shallow else np.nan
        rec.update(pcts)
        results_by_thr[thr].append(rec)

    gdal.Unlink(dem_out); gdal.Unlink(bt_out); gdal.Unlink(lm_out)
    del dem_arr, bt_arr, lm_arr, valid

def add_total(df, thr):
    km2_cols = [c for c in df.columns if c.endswith('_km2')]
    tot = df[km2_cols].sum()
    row = {'ID': -1, 'Navn': 'Norge (sum)', **tot.to_dict()}
    total_key = f'above_{int(thr)}m_total_km2'
    T = tot[total_key]
    for lab in list(BT_LABEL.values()) + list(LM_LABEL.values()):
        row[f'{lab}_pct_of_above_{int(thr)}m'] = 100 * tot[f'{lab}_km2'] / T if T else np.nan
    return pd.concat([df, pd.DataFrame([row])], ignore_index=True)

results_dfs = {thr: add_total(pd.DataFrame(results_by_thr[thr]), thr) for thr in DEPTH_THRESHOLDS}
results_dfs[25.0]

--- Norskehavet ---
  substrat features in bbox: 286128
--- Skagerrak ---
  substrat features in bbox: 38239
--- Nordsjøen ---
  substrat features in bbox: 115283
--- Barentshavet ---
  substrat features in bbox: 60543


,ID,Navn,above_25m_total_km2,losbunn_km2,hardbunn_km2,dk-abcd_km2,dk-efgy_km2,dk-0_km2,losbunn_pct_of_above_25m,hardbunn_pct_of_above_25m,dk-abcd_pct_of_above_25m,dk-efgy_pct_of_above_25m,dk-0_pct_of_above_25m
0,0,Norskehavet,10220.9250,4119.1575,6093.0925,2565.2975,5673.1175,1973.8350,40.301220,59.613905,25.098487,55.504932,19.311706
1,0,Skagerrak,1114.5175,539.2500,571.5675,408.8950,543.5375,158.3850,48.384166,51.283852,36.688074,48.768862,14.211082
2,0,Nordsjøen,1585.0475,423.0500,1158.4375,289.1275,1116.1250,176.2350,26.690052,73.085349,18.240936,70.415871,11.118594
3,0,Barentshavet,1411.0625,780.3025,629.8400,377.8175,572.5575,459.7675,55.298933,44.635868,26.775391,40.576339,32.583071
4,-1,Norge (sum),14331.5525,5861.7600,8452.9375,3641.1375,7905.3375,2768.2225,40.901082,58.981311,25.406441,55.160371,19.315580


In [17]:
results_dfs[30.0]

,ID,Navn,above_30m_total_km2,losbunn_km2,hardbunn_km2,dk-abcd_km2,dk-efgy_km2,dk-0_km2,losbunn_pct_of_above_30m,hardbunn_pct_of_above_30m,dk-abcd_pct_of_above_30m,dk-efgy_pct_of_above_30m,dk-0_pct_of_above_30m
0,0,Norskehavet,11882.9000,4892.5875,6981.1175,3024.4875,6495.5675,2353.6500,41.173346,58.749274,25.452436,54.663150,19.807034
1,0,Skagerrak,1292.1075,637.8925,650.1900,474.5525,618.8375,194.6925,49.368377,50.320117,36.727014,47.893654,15.067825
2,0,Nordsjøen,1927.8100,534.4200,1389.4000,362.0975,1336.2775,225.4450,27.721612,72.071418,18.782842,69.315830,11.694358
3,0,Barentshavet,1797.2500,1048.5675,747.5275,526.4475,680.3750,589.2725,58.342885,41.592850,29.291835,37.856447,32.787453
4,-1,Norge (sum),16900.0675,7113.4675,9768.2350,4387.5850,9131.0575,3363.0600,42.091356,57.799976,25.961938,54.029710,19.899684


In [18]:
from IPython.display import Markdown, display

def to_md(df, title):
    d = df.copy()
    for c in d.select_dtypes('number').columns:
        d[c] = d[c].map(lambda v: f'{v:.1f}' if pd.notna(v) else '')
    if 'ID' in d.columns:
        d = d.drop(columns=['ID'])
    return f'### {title}\n\n' + d.to_markdown(index=False)

for thr, df in results_dfs.items():
    md = to_md(df, f'Økoregioner — areal ≤ {int(thr)} m')
    display(Markdown(md))
    (REPO / f'notebooks/scratch/areal_above_{int(thr)}m_bunntype_lmdk.md').write_text(md, encoding='utf-8')

### Økoregioner — areal ≤ 25 m

| Navn         |   above_25m_total_km2 |   losbunn_km2 |   hardbunn_km2 |   dk-abcd_km2 |   dk-efgy_km2 |   dk-0_km2 |   losbunn_pct_of_above_25m |   hardbunn_pct_of_above_25m |   dk-abcd_pct_of_above_25m |   dk-efgy_pct_of_above_25m |   dk-0_pct_of_above_25m |
|:-------------|----------------------:|--------------:|---------------:|--------------:|--------------:|-----------:|---------------------------:|----------------------------:|---------------------------:|---------------------------:|------------------------:|
| Norskehavet  |               10220.9 |        4119.2 |         6093.1 |        2565.3 |        5673.1 |     1973.8 |                       40.3 |                        59.6 |                       25.1 |                       55.5 |                    19.3 |
| Skagerrak    |                1114.5 |         539.2 |          571.6 |         408.9 |         543.5 |      158.4 |                       48.4 |                        51.3 |                       36.7 |                       48.8 |                    14.2 |
| Nordsjøen    |                1585   |         423.1 |         1158.4 |         289.1 |        1116.1 |      176.2 |                       26.7 |                        73.1 |                       18.2 |                       70.4 |                    11.1 |
| Barentshavet |                1411.1 |         780.3 |          629.8 |         377.8 |         572.6 |      459.8 |                       55.3 |                        44.6 |                       26.8 |                       40.6 |                    32.6 |
| Norge (sum)  |               14331.6 |        5861.8 |         8452.9 |        3641.1 |        7905.3 |     2768.2 |                       40.9 |                        59   |                       25.4 |                       55.2 |                    19.3 |

### Økoregioner — areal ≤ 30 m

| Navn         |   above_30m_total_km2 |   losbunn_km2 |   hardbunn_km2 |   dk-abcd_km2 |   dk-efgy_km2 |   dk-0_km2 |   losbunn_pct_of_above_30m |   hardbunn_pct_of_above_30m |   dk-abcd_pct_of_above_30m |   dk-efgy_pct_of_above_30m |   dk-0_pct_of_above_30m |
|:-------------|----------------------:|--------------:|---------------:|--------------:|--------------:|-----------:|---------------------------:|----------------------------:|---------------------------:|---------------------------:|------------------------:|
| Norskehavet  |               11882.9 |        4892.6 |         6981.1 |        3024.5 |        6495.6 |     2353.7 |                       41.2 |                        58.7 |                       25.5 |                       54.7 |                    19.8 |
| Skagerrak    |                1292.1 |         637.9 |          650.2 |         474.6 |         618.8 |      194.7 |                       49.4 |                        50.3 |                       36.7 |                       47.9 |                    15.1 |
| Nordsjøen    |                1927.8 |         534.4 |         1389.4 |         362.1 |        1336.3 |      225.4 |                       27.7 |                        72.1 |                       18.8 |                       69.3 |                    11.7 |
| Barentshavet |                1797.2 |        1048.6 |          747.5 |         526.4 |         680.4 |      589.3 |                       58.3 |                        41.6 |                       29.3 |                       37.9 |                    32.8 |
| Norge (sum)  |               16900.1 |        7113.5 |         9768.2 |        4387.6 |        9131.1 |     3363.1 |                       42.1 |                        57.8 |                       26   |                       54   |                    19.9 |

In [19]:
# markdown for norway table (produced etter tile-cellen)
def norway_to_md():
    d = pd.DataFrame(norway_by_thr).T
    d.index.name = 'utvalg'
    for c in d.select_dtypes('number').columns:
        d[c] = d[c].map(lambda v: f'{v:.1f}' if pd.notna(v) else '')
    md = '### Norge totalt\n\n' + d.reset_index().to_markdown(index=False)
    (REPO / 'notebooks/scratch/areal_norge.md').write_text(md, encoding='utf-8')
    return Markdown(md)


In [20]:
results_dfs[25.0].to_markdown(index=False)

'|   ID | Navn         |   above_25m_total_km2 |   losbunn_km2 |   hardbunn_km2 |   dk-abcd_km2 |   dk-efgy_km2 |   dk-0_km2 |   losbunn_pct_of_above_25m |   hardbunn_pct_of_above_25m |   dk-abcd_pct_of_above_25m |   dk-efgy_pct_of_above_25m |   dk-0_pct_of_above_25m |\n|-----:|:-------------|----------------------:|--------------:|---------------:|--------------:|--------------:|-----------:|---------------------------:|----------------------------:|---------------------------:|---------------------------:|------------------------:|\n|    0 | Norskehavet  |              10220.9  |      4119.16  |       6093.09  |      2565.3   |      5673.12  |   1973.84  |                    40.3012 |                     59.6139 |                    25.0985 |                    55.5049 |                 19.3117 |\n|    0 | Skagerrak    |               1114.52 |       539.25  |        571.567 |       408.895 |       543.538 |    158.385 |                    48.3842 |                     51.2839 |     

In [21]:
results_dfs[30.0].to_markdown(index=False)


'|   ID | Navn         |   above_30m_total_km2 |   losbunn_km2 |   hardbunn_km2 |   dk-abcd_km2 |   dk-efgy_km2 |   dk-0_km2 |   losbunn_pct_of_above_30m |   hardbunn_pct_of_above_30m |   dk-abcd_pct_of_above_30m |   dk-efgy_pct_of_above_30m |   dk-0_pct_of_above_30m |\n|-----:|:-------------|----------------------:|--------------:|---------------:|--------------:|--------------:|-----------:|---------------------------:|----------------------------:|---------------------------:|---------------------------:|------------------------:|\n|    0 | Norskehavet  |              11882.9  |      4892.59  |       6981.12  |      3024.49  |      6495.57  |   2353.65  |                    41.1733 |                     58.7493 |                    25.4524 |                    54.6632 |                 19.807  |\n|    0 | Skagerrak    |               1292.11 |       637.893 |        650.19  |       474.553 |       618.837 |    194.692 |                    49.3684 |                     50.3201 |     

In [22]:
for thr, df in results_dfs.items():
    out_csv = REPO / f'notebooks/scratch/areal_above_{int(thr)}m_bunntype_lmdk.csv'
    df.to_csv(out_csv, index=False)
    print('Wrote', out_csv)

Wrote /home/kim/work/marint-naturkart-nivaR/notebooks/scratch/areal_above_25m_bunntype_lmdk.csv
Wrote /home/kim/work/marint-naturkart-nivaR/notebooks/scratch/areal_above_30m_bunntype_lmdk.csv


In [23]:
results_dfs[30.0]

,ID,Navn,above_30m_total_km2,losbunn_km2,hardbunn_km2,dk-abcd_km2,dk-efgy_km2,dk-0_km2,losbunn_pct_of_above_30m,hardbunn_pct_of_above_30m,dk-abcd_pct_of_above_30m,dk-efgy_pct_of_above_30m,dk-0_pct_of_above_30m
0,0,Norskehavet,11882.9000,4892.5875,6981.1175,3024.4875,6495.5675,2353.6500,41.173346,58.749274,25.452436,54.663150,19.807034
1,0,Skagerrak,1292.1075,637.8925,650.1900,474.5525,618.8375,194.6925,49.368377,50.320117,36.727014,47.893654,15.067825
2,0,Nordsjøen,1927.8100,534.4200,1389.4000,362.0975,1336.2775,225.4450,27.721612,72.071418,18.782842,69.315830,11.694358
3,0,Barentshavet,1797.2500,1048.5675,747.5275,526.4475,680.3750,589.2725,58.342885,41.592850,29.291835,37.856447,32.787453
4,-1,Norge (sum),16900.0675,7113.4675,9768.2350,4387.5850,9131.0575,3363.0600,42.091356,57.799976,25.961938,54.029710,19.899684


## Norge totalt

Tre uavhengige beregninger, ikke basert på økoregionpolygonene:

1. **Grunt (≤ 25 m og ≤ 30 m)** — hele DEM prosessert i tiles, rasteriser substrat på DEM-gridet per tile og tell piksler.
2. **Uten dybdefilter** — ren geometrisk areal per `BunnType` fra parquet (rask, ingen raster).
3. **DEM uten dybdefilter** — samme tile-baserte metode som (1), men uten dybde­grense: teller alle gyldige DEM-piksler og gir dermed en direkte sammen­ligning med (1) og (2).

In [24]:
# (1) Norge totalt — tile-basert, ingen cutline. Beregner alle dybde-terskler i én pass.
TILE = 4096
dem_ds = gdal.Open(str(DEM_PATH))
band = dem_ds.GetRasterBand(1)
nx, ny = dem_ds.RasterXSize, dem_ds.RasterYSize
gt = dem_ds.GetGeoTransform()
px_, py_ = gt[1], -gt[5]
nd = band.GetNoDataValue()
srs_wkt = dem_ds.GetProjection()

sub_bt_vsi = '/vsimem/substrat_all_bt.fgb'
sub_lm_vsi = '/vsimem/substrat_all_lm.fgb'
for p in (sub_bt_vsi, sub_lm_vsi):
    try: gdal.Unlink(p)
    except RuntimeError: pass
write_dataframe(substrat[substrat['code_bt'] > 0][['code_bt','geometry']], sub_bt_vsi, driver='FlatGeobuf')
write_dataframe(substrat[substrat['code_lm'] > 0][['code_lm','geometry']], sub_lm_vsi, driver='FlatGeobuf')

counts = {thr: {'shallow': 0, **{c: 0 for c in BT_LABEL}, **{f'lm_{c}': 0 for c in LM_LABEL}}
          for thr in DEPTH_THRESHOLDS}
counts['all'] = {'shallow': 0, **{c: 0 for c in BT_LABEL}, **{f'lm_{c}': 0 for c in LM_LABEL}}

n_tiles = ((nx + TILE - 1) // TILE) * ((ny + TILE - 1) // TILE)
done = 0
for yoff in range(0, ny, TILE):
    h = min(TILE, ny - yoff)
    for xoff in range(0, nx, TILE):
        w = min(TILE, nx - xoff)
        dem_tile = band.ReadAsArray(xoff, yoff, w, h)
        valid = dem_tile != nd
        done += 1
        if not valid.any():
            continue
        minx = gt[0] + xoff * px_
        maxy = gt[3] + yoff * gt[5]
        drv = gdal.GetDriverByName('MEM')
        def rasterize_tile(vsi_path, attr):
            sds = drv.Create('', w, h, 1, gdal.GDT_Byte)
            sds.SetGeoTransform((minx, px_, 0, maxy, 0, -py_))
            sds.SetProjection(srs_wkt)
            sds.GetRasterBand(1).SetNoDataValue(0)
            gdal.Rasterize(sds, vsi_path, attribute=attr, allTouched=False)
            arr = sds.GetRasterBand(1).ReadAsArray()
            return arr
        bt_arr = rasterize_tile(sub_bt_vsi, 'code_bt')
        lm_arr = rasterize_tile(sub_lm_vsi, 'code_lm')
        # per-terskel
        for thr in DEPTH_THRESHOLDS:
            shallow = valid & (dem_tile >= -thr) & (dem_tile <= 0.0)
            counts[thr]['shallow'] += int(shallow.sum())
            for c in BT_LABEL:
                counts[thr][c] += int(np.count_nonzero(shallow & (bt_arr == c)))
            for c in LM_LABEL:
                counts[thr][f'lm_{c}'] += int(np.count_nonzero(shallow & (lm_arr == c)))
        # alle gyldige DEM-piksler (uten dybde­filter)
        counts['all']['shallow'] += int(valid.sum())
        for c in BT_LABEL:
            counts['all'][c] += int(np.count_nonzero(valid & (bt_arr == c)))
        for c in LM_LABEL:
            counts['all'][f'lm_{c}'] += int(np.count_nonzero(valid & (lm_arr == c)))
    print(f'  tiles {done}/{n_tiles}')

gdal.Unlink(sub_bt_vsi); gdal.Unlink(sub_lm_vsi)

def norway_series(key):
    c = counts[key]
    T = c['shallow']
    suffix = f'above_{int(key)}m' if key != 'all' else 'dem_all'
    s = {f'{suffix}_total_km2': T * to_km2}
    pcts = {}
    for code, lab in BT_LABEL.items():
        n = c[code]
        s[f'{lab}_km2'] = n * to_km2
        pcts[f'{lab}_pct_of_{suffix}'] = 100 * n / T if T else np.nan
    for code, lab in LM_LABEL.items():
        n = c[f'lm_{code}']
        s[f'{lab}_km2'] = n * to_km2
        pcts[f'{lab}_pct_of_{suffix}'] = 100 * n / T if T else np.nan
    s.update(pcts)
    return pd.Series(s)

norway_by_thr = {thr: norway_series(thr) for thr in DEPTH_THRESHOLDS}
norway_by_thr['dem_all'] = norway_series('all')
pd.DataFrame(norway_by_thr)

  tiles 6/48
  tiles 12/48
  tiles 18/48
  tiles 24/48
  tiles 30/48
  tiles 36/48
  tiles 42/48
  tiles 48/48


,25.0,30.0,dem_all
above_25m_total_km2,14331.600000,NaN,NaN
above_30m_total_km2,NaN,16900.130000,NaN
dem_all_total_km2,NaN,NaN,93967.420000
dk-0_km2,2768.245000,3363.082500,20425.595000
dk-0_pct_of_above_25m,19.315673,NaN,NaN
dk-0_pct_of_above_30m,NaN,19.899743,NaN
dk-0_pct_of_dem_all,NaN,NaN,21.736890
dk-abcd_km2,3641.145000,4387.597500,45791.095000
dk-abcd_pct_of_above_25m,25.406410,NaN,NaN
dk-abcd_pct_of_above_30m,NaN,25.961916,NaN


In [25]:
# (2) Norge totalt, uten dybdefilter — geometrisk areal per BunnType fra parquet.
area_series = substrat.geometry.area  # m² (CRS er metrisk EPSG:25833)
by_class = area_series.groupby(substrat['BunnType']).sum() / 1e6  # km²
total_km2_all = by_class.sum()

norway_all = pd.Series({
    'total_km2': total_km2_all,
    'losbunn_km2': by_class.get('løsbunn', 0.0),
    'hardbunn_km2': by_class.get('fastbunn', 0.0),
    'losbunn_pct': 100 * by_class.get('løsbunn', 0.0) / total_km2_all if total_km2_all else np.nan,
    'hardbunn_pct': 100 * by_class.get('fastbunn', 0.0) / total_km2_all if total_km2_all else np.nan,
})
print('Norge — alt (uten dybdefilter):')
norway_all

Norge — alt (uten dybdefilter):


total_km2       93675.540776
losbunn_km2     64458.655408
hardbunn_km2    29216.885368
losbunn_pct        68.810551
hardbunn_pct       31.189449
dtype: float64

In [26]:
pd.DataFrame(norway_by_thr)

,25.0,30.0,dem_all
above_25m_total_km2,14331.600000,NaN,NaN
above_30m_total_km2,NaN,16900.130000,NaN
dem_all_total_km2,NaN,NaN,93967.420000
dk-0_km2,2768.245000,3363.082500,20425.595000
dk-0_pct_of_above_25m,19.315673,NaN,NaN
dk-0_pct_of_above_30m,NaN,19.899743,NaN
dk-0_pct_of_dem_all,NaN,NaN,21.736890
dk-abcd_km2,3641.145000,4387.597500,45791.095000
dk-abcd_pct_of_above_25m,25.406410,NaN,NaN
dk-abcd_pct_of_above_30m,NaN,25.961916,NaN


In [27]:
norway_to_md()

### Norge totalt

| utvalg   |   above_25m_total_km2 |   above_30m_total_km2 |   dem_all_total_km2 |   dk-0_km2 |   dk-0_pct_of_above_25m |   dk-0_pct_of_above_30m |   dk-0_pct_of_dem_all |   dk-abcd_km2 |   dk-abcd_pct_of_above_25m |   dk-abcd_pct_of_above_30m |   dk-abcd_pct_of_dem_all |   dk-efgy_km2 |   dk-efgy_pct_of_above_25m |   dk-efgy_pct_of_above_30m |   dk-efgy_pct_of_dem_all |   hardbunn_km2 |   hardbunn_pct_of_above_25m |   hardbunn_pct_of_above_30m |   hardbunn_pct_of_dem_all |   losbunn_km2 |   losbunn_pct_of_above_25m |   losbunn_pct_of_above_30m |   losbunn_pct_of_dem_all |
|:---------|----------------------:|----------------------:|--------------------:|-----------:|------------------------:|------------------------:|----------------------:|--------------:|---------------------------:|---------------------------:|-------------------------:|--------------:|---------------------------:|---------------------------:|-------------------------:|---------------:|----------------------------:|----------------------------:|--------------------------:|--------------:|---------------------------:|---------------------------:|-------------------------:|
| 25.0     |               14331.6 |                       |                     |     2768.2 |                    19.3 |                         |                       |        3641.1 |                       25.4 |                            |                          |        7905.4 |                       55.2 |                            |                          |         8453   |                          59 |                             |                           |        5861.8 |                       40.9 |                            |                          |
| 30.0     |                       |               16900.1 |                     |     3363.1 |                         |                    19.9 |                       |        4387.6 |                            |                         26 |                          |        9131.1 |                            |                         54 |                          |         9768.3 |                             |                        57.8 |                           |        7113.5 |                            |                       42.1 |                          |
| dem_all  |                       |                       |             93967.4 |    20425.6 |                         |                         |                  21.7 |       45791.1 |                            |                            |                     48.7 |       27436.8 |                            |                            |                     29.2 |        29208.9 |                             |                             |                      31.1 |       64444.6 |                            |                            |                     68.6 |

In [29]:
# Skriv GeoTIFF med LM_DK-klasse for piksler ≤ 25 m dyp, samme pikselverdier som modell:
#   0 = DK_AB,C,D (løsbunn), 1 = DK_0 (blanding), 2 = DK_EFGY (fastbunn), 255 = nodata.
# Mønster fra notebooks/01_substrat/03_predict.ipynb: bygg hele arrayen i RAM og skriv med rasterio.
import rasterio as rio
from rasterio.transform import Affine

OUT_TIF = REPO / 'notebooks/scratch/lmdk_above_25m_norge.tif'
THR = 25.0
NODATA = 255
REMAP = np.full(256, NODATA, dtype=np.uint8)
REMAP[1] = 0  # DK_AB, DK_C, DK_D → løsbunn
REMAP[2] = 2  # DK_EFGY         → fastbunn
REMAP[3] = 1  # DK_0            → blanding

dem_ds = gdal.Open(str(DEM_PATH))
band = dem_ds.GetRasterBand(1)
nx, ny = dem_ds.RasterXSize, dem_ds.RasterYSize
gt = dem_ds.GetGeoTransform()
nd = band.GetNoDataValue()
srs_wkt = dem_ds.GetProjection()

print('Rasteriserer LM_DK til hele landet ...')
sub_lm_vsi = '/vsimem/substrat_lm_out.fgb'
try: gdal.Unlink(sub_lm_vsi)
except RuntimeError: pass
write_dataframe(substrat[substrat['code_lm'] > 0][['code_lm','geometry']], sub_lm_vsi, driver='FlatGeobuf')

mem = gdal.GetDriverByName('MEM').Create('', nx, ny, 1, gdal.GDT_Byte)
mem.SetGeoTransform(gt); mem.SetProjection(srs_wkt)
mem.GetRasterBand(1).SetNoDataValue(0)
gdal.Rasterize(mem, sub_lm_vsi, attribute='code_lm', allTouched=False)
lm_map = mem.GetRasterBand(1).ReadAsArray()
mem = None; gdal.Unlink(sub_lm_vsi)

print('Leser DEM ...')
dem_arr = band.ReadAsArray()
shallow = (dem_arr != nd) & (dem_arr >= -THR) & (dem_arr <= 0.0)
print(f'  ≤{int(THR)} m piksler: {int(shallow.sum()):,}')

pred_map = REMAP[lm_map]
pred_map[~shallow] = NODATA
print(f'  klassifiserte piksler skrevet: {int((pred_map != NODATA).sum()):,}')

transform = Affine.from_gdal(*gt)
with rio.open(
    OUT_TIF, 'w',
    driver='GTiff',
    height=ny, width=nx,
    count=1, dtype=np.uint8,
    crs=srs_wkt,
    transform=transform,
    nodata=NODATA,
    compress='deflate',
    tiled=True, blockxsize=512, blockysize=512,
) as dst:
    dst.write(pred_map, 1)
    dst.write_colormap(1, {
        0: (166, 206, 227, 255),   # løsbunn / DK_AB,C,D
        1: (178, 223, 138, 255),   # blanding / DK_0
        2: (31, 120, 180, 255),    # fastbunn / DK_EFGY
        NODATA: (0, 0, 0, 0),
    })
print('Wrote', OUT_TIF)

Rasteriserer LM_DK til hele landet ...
Leser DEM ...
  ≤25 m piksler: 5,732,640
  klassifiserte piksler skrevet: 5,725,898
Wrote /home/kim/work/marint-naturkart-nivaR/notebooks/scratch/lmdk_above_25m_norge.tif
